> ⚠️ **This notebook needs a real control compartment to render meaningfully.**
> The CellANOVA backend requires `adata.uns['control_dict']` mapping a
> pool name → list of batch labels considered controls (cells expected to
> be biologically homogeneous across batches). On a pbmc3k synthetic
> 2-batch demo with no real control compartment, the variance-decomposition
> step produces shape-mismatched outputs and the notebook does not render.
>
> On a real dataset with a designated control compartment (e.g. peripheral
> blood across treatment arms where naive T cells are expected to be
> stable), the recipe below works end to end.


# Batch correction with CellANOVA

CellANOVA (Zhang et al., *Nat Biotechnol* 2024) is a **variance-decomposition** approach. Given a control pool of cells expected to be biologically homogeneous across batches, it estimates per-batch nuisance variance and removes it from the rest of the data. Especially strong when you can designate a control compartment that shouldn't change across conditions.

Set `adata.uns['control_dict']` to a dict mapping pool name → list of batch labels **before** the call. In this demo we use both synthetic batches as a single "all" pool, which shows the API surface but is not biologically meaningful — in production, designate a real control compartment.

This is one of the **omicverse batch-correction zoo** tutorials. See [batch/index](../index.md) for the overview / decision tree, or [../t_single_batch](../t_single_batch.ipynb) for the side-by-side comparison of every backend on a real benchmark.

## Load a 2-batch demo from pbmc3k

We use the canonical 10x pbmc3k dataset and synthesise a 2-batch label by random assignment, then plant a gene-shift on `batch_B` so the uncorrected UMAP shows a visible batch effect. This keeps the notebook self-contained and fast (~2 min end-to-end) — for a real multi-donor benchmark with [scib-metrics] scoring, see [../t_single_batch](../t_single_batch.ipynb).

In [ ]:
import omicverse as ov
import scanpy as sc
import numpy as np
import pandas as pd

# Load the cached pbmc3k 10x raw counts (~2700 cells x ~32000 genes).
adata = ov.datasets.pbmc3k(processed=False)
adata.var_names_make_unique()
adata.obs_names_make_unique()
adata.layers['counts'] = adata.X.copy()  # scvi-tools needs raw counts here

# Synthesise a 2-batch demo: alternate cells, then plant a gene-shift on
# batch B so the uncorrected UMAP visibly separates by batch.
rng = np.random.default_rng(0)
batch = rng.choice(['batch_A', 'batch_B'], size=adata.n_obs, p=[0.5, 0.5])
adata.obs['batch'] = pd.Categorical(batch)

# Inject a multiplicative effect on the first 500 genes for batch_B.
from scipy.sparse import issparse, csr_matrix
X = adata.X.toarray() if issparse(adata.X) else adata.X.copy()
is_B = (adata.obs['batch'] == 'batch_B').values
X[is_B, :500] = X[is_B, :500] * 2.0
adata.X = csr_matrix(X)
adata.layers['counts'] = adata.X.copy()
adata

## Preprocess + PCA + cluster

Same QC → HVG-pearson → log-norm → PCA pipeline shared across every backend in the zoo. A quick Leiden cluster gives a synthetic `celltype` label that scANVI / scPoli can use as a prototype anchor.

In [ ]:
# Standard omicverse preprocess (QC → HVG-via-pearson → log-norm → PCA).
adata = ov.pp.qc(adata, tresh={'mito_perc': 0.2, 'nUMIs': 500,
                                 'detected_genes': 250})
ov.utils.store_layers(adata, layers='counts')
adata = ov.pp.preprocess(adata, mode='shiftlog|pearson', n_HVGs=2000,
                         batch_key=None)
adata.raw = adata
adata = adata[:, adata.var.highly_variable_features].copy()
ov.pp.scale(adata)
ov.pp.pca(adata, layer='scaled', n_pcs=30)

# Quick Leiden cluster to use as a synthetic celltype label for scANVI etc.
sc.pp.neighbors(adata, use_rep='scaled|original|X_pca', n_neighbors=15)
sc.tl.leiden(adata, resolution=0.5, flavor='igraph', directed=False,
             n_iterations=2)
adata.obs['celltype'] = adata.obs['leiden'].astype(str).map(
    lambda c: f'cluster_{c}'
).astype('category')
adata

## Uncorrected baseline

The planted batch effect is visible in the uncorrected UMAP:

In [ ]:
# Pre-correction UMAP shows the planted batch effect.
sc.tl.umap(adata, min_dist=0.3)
adata.obsm['X_umap_uncorrected'] = adata.obsm['X_umap'].copy()
ov.pl.embedding(adata, basis='X_umap_uncorrected',
                color=['batch', 'celltype'],
                frameon='small', wspace=0.5)

## Run `ov.single.batch_correction(methods='cellanova')`

For the scvi-tools family backends, the wrapper auto-routes `**kwargs` between the model's `__init__` (architecture) and `.train()` (optimisation) destinations. See the **Key parameters** section below.

In [ ]:
# Control-pool setup: maps pool name → list of batch labels.
# Replace with your actual control compartment in production.
adata.uns['control_dict'] = {
    'all': list(adata.obs['batch'].cat.categories),
}

ov.single.batch_correction(
    adata,
    batch_key='batch',
    methods='CellANOVA',
    n_pcs=30,
)

## Corrected embedding

Every backend writes its corrected representation to a stable obsm key — for this one it is `adata.obsm['X_cellanova']`. We project via `ov.utils.mde` for a lightweight UMAP-style display.

In [ ]:
adata.obsm['X_mde_cellanova'] = ov.utils.mde(adata.obsm['X_cellanova'])
ov.pl.embedding(
    adata,
    basis='X_mde_cellanova',
    color=['batch', 'celltype'],
    frameon='small',
    wspace=0.5,
)

## Key parameters

- `adata.uns['control_dict']` — REQUIRED. Dict mapping pool name → list of batch labels considered controls.
- CellANOVA writes a denoised expression matrix into `adata.layers['denoised']` alongside the embedding.


## Related tutorials

- [t_batch_combat](t_batch_combat.ipynb) — when no control pool is available.
- [t_batch_harmony](t_batch_harmony.ipynb) — embedding-level only.

For the full side-by-side comparison with scib-metrics scoring, see [../t_single_batch](../t_single_batch.ipynb).